# ĐỒ ÁN MÁY HỌC: TIỀN XỬ LÝ DỮ LIỆU SKU (STOCK KEEPING UNITS)
**Mục tiêu:** Tiền xử lý dữ liệu chuẩn bị cho bài toán phân cụm (**K-Means, DBSCAN, đặc biệt là Incremental DBSCAN**).  
**Tập dữ liệu:** `sku_data.xlsx` (Nguồn: Trialto Latvia LTD - UCI ML Repository).

---
### Tóm tắt pipeline tiền xử lý đề xuất:
1. **Khám phá ban đầu (EDA & Data Audit):** Kiểm tra missing values, kiểu dữ liệu, phân phối, giá trị 0 và outliers.
2. **Lọc/Tách ID:** Tách cột `ID` thành index định danh.
3. **Xử lý bất thường biến `Expire date`:**
   - Hiệu chỉnh lỗi nhập liệu: gộp `734` ngày thành `730` ngày (2 năm).
   - Mã hóa phân nhóm thời hạn (Binning Ordinal / Shelf-life Categorization) để chuẩn hóa bước nhảy khoảng cách Euclidean.
4. **Phân loại nhóm "Ghost SKU / Dead Stock":**
   - Đánh dấu 206 SKU có toàn bộ đặc trưng số bằng 0 (chỉ tồn tại trên hệ thống nhưng không có giao dịch/vật lý).
5. **Biến đổi giảm độ lệch (Log Transformation):**
   - Áp dụng `log1p(x) = ln(x + 1)` cho các biến lệch cực đoan (`Unitprice`, `Outbound number`, `Total outbound`, `Units per pal`).
6. **Kỹ thuật đặc trưng (Feature Engineering):**
   - Tạo biến `Avg_units_per_order` = `Total outbound` / `Outbound number` (cho đơn vị có xuất hàng) phản ánh quy mô đơn hàng.
7. **Chuẩn hóa dữ liệu (Robust Scaling):**
   - Dùng `RobustScaler` (dựa trên Median & IQR) nhằm giữ vững tính cấu trúc của cụm mà không bị ảnh hưởng bởi các outliers cực đoan.
8. **Xuất file sạch:** Lưu kết quả sẵn sàng nạp trực tiếp vào các mô hình phân cụm (batch và incremental).


In [ ]:
# 1. Khai báo các thư viện cần thiết
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler

# Cấu hình hiển thị
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.4f' % x)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Import thư viện thành công!")


In [ ]:
# 2. Đọc tập dữ liệu gốc
file_path = r"D:\HP Máy học\Đồ án\sku_data.xlsx"
df_raw = pd.read_excel(file_path)

print(f"Kích thước tập dữ liệu gốc: {df_raw.shape[0]} dòng, {df_raw.shape[1]} cột")
df_raw.head()


## Bước 1: Khám phá ban đầu (EDA & Thống kê mô tả)
Kiểm tra cấu trúc, giá trị null thực tế và tỷ lệ giá trị 0.


In [ ]:
# Kiểm tra thông tin chung và tỷ lệ giá trị 0
info_df = pd.DataFrame({
    'Kiểu dữ liệu': df_raw.dtypes,
    'Số lượng Null': df_raw.isnull().sum(),
    'Tỷ lệ Null (%)': (df_raw.isnull().sum() / len(df_raw)) * 100,
    'Số lượng giá trị 0': (df_raw == 0).sum(),
    'Tỷ lệ giá trị 0 (%)': ((df_raw == 0).sum() / len(df_raw)) * 100,
    'Số giá trị duy nhất': df_raw.nunique()
})
info_df


## Bước 2: Tách biệt cột định danh `ID`
Cột `ID` chỉ mang tính chất định danh SKU, không tham gia vào việc đo lường khoảng cách hình học trong không gian đặc trưng.


In [ ]:
df = df_raw.copy()
sku_ids = df['ID']
df = df.drop(columns=['ID'])
print("Đã tách ID. Các đặc trưng còn lại:", list(df.columns))


## Bước 3: Xử lý biến `Expire date` (Hạn sử dụng)
- Thống kê các mức hạn sử dụng gốc.
- Sửa giá trị bất thường: `734` ngày (chỉ 1 dòng duy nhất, lệch so với mốc 2 năm = 730 ngày) -> gộp về `730`.
- Tạo biến phân nhóm hạn sử dụng `Expire_category`:
  - `0`: Không có hạn sử dụng (hàng phi thực phẩm / thiết bị / phụ kiện).
  - `1`: Ngắn hạn (~1 năm = 365 ngày).
  - `2`: Trung hạn (~1.25 - 1.5 năm = 456 - 547 ngày).
  - `3`: Dài hạn (~2 năm = 730 ngày).


In [ ]:
print("Tần suất Expire date trước khi hiệu chỉnh:")
print(df['Expire date'].value_counts().sort_index())

# 1. Gộp giá trị 734 về 730
df['Expire date'] = df['Expire date'].replace({734: 730})

# 2. Tạo cột mã hóa ordinal cho nhóm vòng đời hàng hóa
def categorize_shelf_life(days):
    if days == 0:
        return 0 # Không có hạn (Non-perishable)
    elif days <= 365:
        return 1 # Ngắn hạn (1 năm)
    elif days <= 547:
        return 2 # Trung hạn (1.5 năm)
    else:
        return 3 # Dài hạn (2 năm trở lên)

df['Expire_category'] = df['Expire date'].apply(categorize_shelf_life)

print("
Phân bổ sau khi gộp và phân nhóm:")
print(df[['Expire date', 'Expire_category']].value_counts().sort_index())


## Bước 4: Nhận diện và gắn nhãn nhóm "Ghost SKU / Dead Stock"
Trong dữ liệu có các mặt hàng có toàn bộ các thông số kinh tế (`Unitprice`), lưu thông (`Outbound number`, `Total outbound`) và thông số kích thước vật lý (`Pal grossweight`, `Pal height`, `Units per pal`) đều bằng 0.
- Về góc độ kinh doanh: Đây là SKU "ma" hoặc mới khai báo trên hệ thống WMS mà chưa từng lưu kho vật lý hay phát sinh luân chuyển.
- Việc gắn cờ (`Is_ghost_sku`) giúp ta có thể tách riêng thành 1 cụm cố định hoặc lọc ra khi đánh giá mô hình.


In [ ]:
feature_cols = ['Unitprice', 'Outbound number', 'Total outbound', 'Pal grossweight', 'Pal height', 'Units per pal']
df['Is_ghost_sku'] = (df[feature_cols] == 0).all(axis=1).astype(int)

ghost_count = df['Is_ghost_sku'].sum()
print(f"Số lượng Ghost SKU / Dead stock hoàn toàn: {ghost_count} ({ghost_count / len(df) * 100:.2f}%)")


## Bước 5: Kỹ thuật đặc trưng (Feature Engineering)
Tạo đặc trưng `Avg_units_per_order` phản ánh quy mô trung bình mỗi lần xuất hàng:
$$\text{Avg\_units\_per\_order} = \frac{\text{Total outbound}}{\text{Outbound number}}$$
(Đối với SKU chưa từng xuất kho, giá trị được gán là 0).


In [ ]:
# Tính toán quy mô xuất trung bình trên mỗi đơn xuất
df['Avg_units_per_order'] = np.where(
    df['Outbound number'] > 0,
    df['Total outbound'] / df['Outbound number'],
    0.0
)

print("Mô tả biến mới Avg_units_per_order:")
print(df['Avg_units_per_order'].describe())


## Bước 6: Phân tích độ lệch (Skewness) & Biến đổi Log-Transformation
Do đặc thù ngành kho bãi, các chỉ số về giá cả, số lượt xuất kho và số lượng đơn vị/pallet có phân phối lệch phải cực kỳ nặng (Right-skewed với Skewness > 4, thậm chí > 25).
Áp dụng phép biến đổi:
$$x_{log} = \ln(x + 1)$$


In [ ]:
skew_cols = ['Unitprice', 'Outbound number', 'Total outbound', 'Units per pal', 'Avg_units_per_order']

print("=== HỆ SỐ LỆCH (SKEWNESS) TRƯỚC VÀ SAU BIẾN ĐỔI LOG1P ===")
skew_comparison = []

for col in skew_cols:
    skew_before = df[col].skew()
    log_transformed = np.log1p(df[col])
    skew_after = log_transformed.skew()
    skew_comparison.append({
        'Đặc trưng': col,
        'Skew trước': skew_before,
        'Skew sau log1p': skew_after
    })
    # Cập nhật trực tiếp giá trị log vào cột tương ứng
    df[col] = log_transformed

pd.DataFrame(skew_comparison)


## Bước 7: Trực quan hóa tương quan sau tiền xử lý
Kiểm tra lại ma trận tương quan giữa các đặc trưng sau khi đã log transform.


In [ ]:
# Danh sách các biến số sẽ đưa vào mô hình phân cụm
model_features = [
    'Unitprice', 
    'Expire_category', 
    'Outbound number', 
    'Total outbound', 
    'Pal grossweight', 
    'Pal height', 
    'Units per pal',
    'Avg_units_per_order'
]

plt.figure(figsize=(9, 7))
corr_matrix = df[model_features].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Ma trận tương quan giữa các đặc trưng (sau khi Log Transform & Encoding)")
plt.tight_layout()
plt.show()


## Bước 8: Chuẩn hóa dữ liệu bằng `RobustScaler`
- Thay vì dùng `StandardScaler` (rất nhạy cảm với cực trị), ta sử dụng `RobustScaler` (sử dụng Median và Interquartile Range - IQR) để co giãn các trục đặc trưng về cùng thang đo.
- Điều này cực kỳ quan trọng cho các giải thuật tính khoảng cách như K-Means, DBSCAN và Incremental DBSCAN.


In [ ]:
scaler = RobustScaler()
X_scaled = scaler.fit_transform(df[model_features])

# Tạo dataframe chứa dữ liệu đã chuẩn hóa hoàn chỉnh
df_scaled = pd.DataFrame(X_scaled, columns=[f"{col}_scaled" for col in model_features])
df_scaled.insert(0, 'ID', sku_ids)
df_scaled['Is_ghost_sku'] = df['Is_ghost_sku']

print("Kích thước ma trận đặc trưng sau chuẩn hóa:", df_scaled.shape)
df_scaled.head()


## Bước 9: Lưu trữ dữ liệu sạch đầu ra
Xuất tập dữ liệu đã qua tiền xử lý ra file `.xlsx` và `.csv` để phục vụ trực tiếp cho quá trình huấn luyện mô hình phân cụm (Batch & Incremental).


In [ ]:
output_clean_excel = r"D:\HP Máy học\Đồ án\sku_data_preprocessed.xlsx"
output_clean_csv = r"D:\HP Máy học\Đồ án\sku_data_preprocessed.csv"

df_scaled.to_excel(output_clean_excel, index=False)
df_scaled.to_csv(output_clean_csv, index=False)

print(f"Đã lưu thành công dữ liệu tiền xử lý tại:")
print(f" - Excel: {output_clean_excel}")
print(f" - CSV: {output_clean_csv}")
